## Setup — imports & config

In [12]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import mlflow

DATA_DIR = Path("training")
WINDOW_SIZE = 6

# labs rarely ordered — presence itself is a signal, not just the value
sparse_labs = ["TroponinI", "Bilirubin_direct", "Fibrinogen", "Bilirubin_total", "Alkalinephos", "AST"]
# static per-patient facts + the label — never forward-filled or median-filled
full_filled_cols = ["SepsisLabel", "Gender", "Age", "HospAdmTime", "ICULOS"]
# 100% missing across the dataset — dead column
empty_cols = ["EtCO2"]

## Load file lists

`training_setA` and `training_setB` come from two *different hospital systems*
(confirmed in the Challenge paper). We use A as train+validation and B as a fully
held-out test set — this gives a genuine cross-hospital generalization check with
zero leakage risk, since no patient can possibly appear in both.

In [13]:
def load_patient_files(set_name):
    """Return all .psv file paths for a training set folder, e.g. 'training_setA'."""
    return list((DATA_DIR / set_name).glob("*.psv"))

files_a = load_patient_files("training_setA")   # train + validation
files_b = load_patient_files("training_setB")   # held-out test
print(f"{len(files_a)} files in set A (train/val), {len(files_b)} files in set B (test)")

20336 files in set A (train/val), 20000 files in set B (test)


## Quick look at one patient — raw, unfilled, just to eyeball the shape of the data

In [14]:
sample_path = DATA_DIR / "training_setA" / "p020643.psv"
sample_df = pd.read_csv(sample_path, sep="|")
print(sample_df.shape)
sample_df[["HR", "O2Sat", "SBP", "SepsisLabel"]].head(10)

(33, 41)


,HR,O2Sat,SBP,SepsisLabel
0,94.0,99.0,167.0,0
1,82.0,99.0,108.0,0
2,80.0,100.0,115.0,0
3,79.0,100.0,176.0,0
4,86.0,98.0,187.0,0
5,86.0,98.0,NaN,0
6,88.0,99.0,NaN,0
7,96.0,NaN,NaN,0
8,90.0,97.0,NaN,0
9,84.0,NaN,NaN,0


## Concatenate raw training-hospital data only

Used to compute population-level fallback stats. Deliberately built from `files_a`
only — set B is the held-out test hospital, so its statistics must never leak into
anything the pipeline was fit on, same principle as fitting a scaler on train only.

In [15]:
def load_raw_concat(files):
    """Read and concatenate raw (unfilled) rows from a list of patient files."""
    return pd.concat([pd.read_csv(f, sep="|") for f in files], ignore_index=True)

raw_train = load_raw_concat(files_a)
print(raw_train.shape)

(790215, 41)


## Missingness per column, raw (training hospital only) — confirms the fill-tier decisions made earlier

In [16]:
def missing_report(df):
    """% missing per column, worst first."""
    return (df.isna().mean() * 100).sort_values(ascending=False).round(1)

missing_report(raw_train)

EtCO2               100.0
TroponinI            99.9
Bilirubin_direct     99.9
Fibrinogen           99.2
Bilirubin_total      98.8
Alkalinephos         98.5
AST                  98.5
Lactate              96.6
PTT                  95.2
SaO2                 95.0
Calcium              95.0
Phosphate            95.0
Platelets            93.5
Creatinine           93.4
WBC                  92.5
Magnesium            92.2
HCO3                 91.9
BUN                  91.8
Chloride             91.7
PaCO2                91.2
Hgb                  91.2
BaseExcess           89.6
Potassium            89.1
pH                   88.5
Hct                  88.2
Glucose              87.8
FiO2                 85.8
Temp                 66.2
Unit1                48.9
Unit2                48.9
DBP                  48.1
SBP                  15.2
O2Sat                12.0
MAP                  10.2
Resp                  9.8
HR                    7.7
HospAdmTime           0.0
Gender                0.0
Age         

## Population median fallback + feature groups

Median is computed once, from set A only, reused for every patient in both A and B.

Features split into two groups for windowing:
- **`WINDOW_COLS`** — vitals + labs, values that genuinely vary hour to hour. These get
  mean/min/max/trend aggregated over each window.
- **`PASSTHROUGH_COLS`** — static per-patient facts (age, gender, ICU unit), the
  `_ever_measured` flags, and `hours_since_lactate`. Aggregating a constant or a
  recency counter with mean/min/max/trend would just produce noise (trend of a
  constant is always 0) — these are taken as-is at the window's last hour instead.

In [17]:
fill_cols = [c for c in raw_train.columns if c not in full_filled_cols and c not in empty_cols]
population_median = raw_train[fill_cols].median()

WINDOW_COLS = [c for c in fill_cols if c not in ("Unit1", "Unit2")]
PASSTHROUGH_COLS = (
    [c for c in full_filled_cols if c != "SepsisLabel"]
    + ["Unit1", "Unit2"]
    + [f"{c}_ever_measured" for c in sparse_labs]
    + ["hours_since_lactate"]
)
print(f"{len(WINDOW_COLS)} windowed columns, {len(PASSTHROUGH_COLS)} passthrough columns")

33 windowed columns, 13 passthrough columns


## Per-patient cleaning

Drop dead columns, flag rarely-measured labs *before* filling (the flag is what lets
the model tell "measured near-median" apart from "never tested, filled with median"),
forward-fill within the patient, then fall back to the population median for any
leading gap forward-fill can't reach (nothing measured yet at the start of a stay).

In [18]:
def clean_patient(path, population_median):
    """Load one patient's .psv file and return a fully filled dataframe."""
    df = pd.read_csv(path, sep="|")
    df = df.drop(columns=empty_cols, errors="ignore")

    cols = [c for c in df.columns if c not in full_filled_cols]
    was_missing = df[cols].isna()

    for col in sparse_labs:
        df[f"{col}_ever_measured"] = int((~was_missing[col]).any())

    df[cols] = df[cols].ffill()
    df[cols] = df[cols].fillna(population_median[cols])

    # recency feature: hours since the last real lactate reading
    df["hours_since_lactate"] = was_missing["Lactate"].groupby(
        (~was_missing["Lactate"]).cumsum()
    ).cumcount()

    return df

## Sliding-window feature builder

Turns an hourly sequence into fixed-size rolling windows. `window_cols` get
mean/min/max/trend; `passthrough_cols` are taken as-is at the window's last hour.

**Vectorized with `pandas.rolling()`** instead of a manual per-hour Python loop —
the original version called `.mean()/.min()/.max()` separately for every column on
every single hour (~120 small pandas calls per hour, per patient), which is what
made the full-dataset run take so long. `rolling()` computes each stat across the
whole column in one C-level pass instead.

In [19]:
def build_windows(df, window_cols, passthrough_cols, window=WINDOW_SIZE):
    """One row per valid window, each labeled with SepsisLabel at the window's last hour."""
    if len(df) < window:
        cols = ["label"] + [f"{c}_{stat}" for c in window_cols for stat in ("mean", "min", "max", "trend")] + list(passthrough_cols)
        return pd.DataFrame(columns=cols)

    roll = df[window_cols].rolling(window=window)
    means = roll.mean()
    mins = roll.min()
    maxs = roll.max()
    trend = df[window_cols] - df[window_cols].shift(window - 1)

    valid_idx = df.index[window - 1:]

    # build as a plain dict first, then construct the DataFrame once —
    # assigning columns one at a time into an existing frame triggers a
    # reallocation per column (pandas' "highly fragmented" warning)
    data = {"label": df.loc[valid_idx, "SepsisLabel"].values}
    for col in window_cols:
        data[f"{col}_mean"] = means.loc[valid_idx, col].values
        data[f"{col}_min"] = mins.loc[valid_idx, col].values
        data[f"{col}_max"] = maxs.loc[valid_idx, col].values
        data[f"{col}_trend"] = trend.loc[valid_idx, col].values
    for col in passthrough_cols:
        data[col] = df.loc[valid_idx, col].values

    return pd.DataFrame(data)

## Sanity check on one patient before scaling to the full dataset

In [20]:
test_df = clean_patient(files_a[0], population_median)
print(test_df.isna().sum().sum(), "total remaining NaNs")

test_windows = build_windows(test_df, WINDOW_COLS, PASSTHROUGH_COLS, WINDOW_SIZE)
print(test_windows.shape)
test_windows.head()

0 total remaining NaNs
(50, 146)


,label,HR_mean,HR_min,HR_max,HR_trend,O2Sat_mean,O2Sat_min,O2Sat_max,O2Sat_trend,Temp_mean,...,ICULOS,Unit1,Unit2,TroponinI_ever_measured,Bilirubin_direct_ever_measured,Fibrinogen_ever_measured,Bilirubin_total_ever_measured,Alkalinephos_ever_measured,AST_ever_measured,hours_since_lactate
0,0,77.333333,74.0,80.0,-4.0,100.000000,100.0,100.0,0.0,36.216667,...,8,0,1,0,0,1,1,1,1,0
1,0,75.666667,70.0,80.0,-6.0,100.000000,100.0,100.0,0.0,36.150000,...,9,0,1,0,0,1,1,1,1,0
2,0,74.166667,67.0,80.0,-13.0,100.000000,100.0,100.0,0.0,36.175000,...,10,0,1,0,0,1,1,1,1,1
3,0,71.583333,64.5,78.0,-13.5,100.000000,100.0,100.0,0.0,36.225000,...,11,0,1,0,0,1,1,1,1,2
4,0,69.416667,64.5,76.0,-9.0,99.833333,99.0,100.0,-1.0,36.358333,...,12,0,1,0,0,1,1,1,1,3


## Full pipeline

`process_patient` chains cleaning + windowing for one file and tags every row with
`patient_id` (needed for a leak-safe train/validation split within set A).
`build_full_dataset` runs it over a list of files and concatenates the results.

In [21]:
def process_patient(path, population_median, window_cols, passthrough_cols, window=WINDOW_SIZE):
    """Clean one patient's file, window it, and tag rows with patient_id."""
    df = clean_patient(path, population_median)
    windows = build_windows(df, window_cols, passthrough_cols, window)
    windows.insert(0, "patient_id", path.stem)
    return windows

def build_full_dataset(files, population_median, window_cols, passthrough_cols, window=WINDOW_SIZE):
    """Run process_patient over every file and concatenate into one table."""
    processed = [process_patient(f, population_median, window_cols, passthrough_cols, window) for f in files]
    return pd.concat(processed, ignore_index=True)

## Sample run — confirms the full pipeline works end-to-end before the expensive full run

In [22]:
sample_dataset = build_full_dataset(files_a[:200], population_median, WINDOW_COLS, PASSTHROUGH_COLS)
print(sample_dataset.shape)
sample_dataset.head()

(6523, 147)


,patient_id,label,HR_mean,HR_min,HR_max,HR_trend,O2Sat_mean,O2Sat_min,O2Sat_max,O2Sat_trend,...,ICULOS,Unit1,Unit2,TroponinI_ever_measured,Bilirubin_direct_ever_measured,Fibrinogen_ever_measured,Bilirubin_total_ever_measured,Alkalinephos_ever_measured,AST_ever_measured,hours_since_lactate
0,p014977,0,77.333333,74.0,80.0,-4.0,100.000000,100.0,100.0,0.0,...,8,0.0,1.0,0,0,1,1,1,1,0
1,p014977,0,75.666667,70.0,80.0,-6.0,100.000000,100.0,100.0,0.0,...,9,0.0,1.0,0,0,1,1,1,1,0
2,p014977,0,74.166667,67.0,80.0,-13.0,100.000000,100.0,100.0,0.0,...,10,0.0,1.0,0,0,1,1,1,1,1
3,p014977,0,71.583333,64.5,78.0,-13.5,100.000000,100.0,100.0,0.0,...,11,0.0,1.0,0,0,1,1,1,1,2
4,p014977,0,69.416667,64.5,76.0,-9.0,99.833333,99.0,100.0,-1.0,...,12,0.0,1.0,0,0,1,1,1,1,3


## Build the real datasets

`train_dataset` — every window from every patient in set A.
`test_dataset` — every window from every patient in set B, fully held out until the end.

This runs over ~40,336 patients — expect it to take a while. Uncomment once the
sample above looks right.

In [29]:
test_dataset = build_full_dataset(files_b, population_median, WINDOW_COLS, PASSTHROUGH_COLS)
print(test_dataset.shape)

(661995, 147)


In [23]:
train_dataset = build_full_dataset(files_a, population_median, WINDOW_COLS, PASSTHROUGH_COLS)

print(train_dataset.shape)


(688535, 147)


## Patient-level train/validation split within set A

`GroupShuffleSplit` keeps every window from one patient entirely in train or entirely
in validation — same leakage concern as before, just within A instead of across A/B.

In [24]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(splitter.split(train_dataset, groups=train_dataset["patient_id"]))
train_split = train_dataset.iloc[train_idx]
val_split = train_dataset.iloc[val_idx]

feature_cols = [c for c in train_dataset.columns if c not in ("patient_id", "label")]
X_train, y_train = train_split[feature_cols], train_split["label"]
X_val, y_val = val_split[feature_cols], val_split["label"]
print(X_train.shape, X_val.shape, y_train.mean(), y_val.mean())

(549773, 145) (138762, 145) 0.022922187884817917 0.022297170695147087


## RF training helper, with MLflow logging

Wraps fit + evaluate + log so every run (baseline, balanced, and anything after) is
tracked from the start, per this month's plan.

In [25]:
def run_rf_experiment(name, model, X_train, y_train, X_val, y_val):
    with mlflow.start_run(run_name=name):
        model.fit(X_train, y_train)
        preds = model.predict(X_val)
        report = classification_report(y_val, preds, output_dict=True)

        mlflow.log_params(model.get_params())
        mlflow.log_metric("recall_pos", report["1"]["recall"])
        mlflow.log_metric("precision_pos", report["1"]["precision"])
        mlflow.log_metric("accuracy", report["accuracy"])

        print(classification_report(y_val, preds))
    return model

## Baseline RF — default settings, same first move as month 3

In [26]:
rf_baseline = run_rf_experiment(
    "rf_baseline",
    RandomForestClassifier(random_state=42, n_jobs=-1),
    X_train, y_train, X_val, y_val,
)

2026/07/10 21:29:35 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/10 21:29:35 INFO mlflow.store.db.utils: Updating database tables


              precision    recall  f1-score   support

           0       0.98      1.00      0.99    135668
           1       0.16      0.00      0.01      3094

    accuracy                           0.98    138762
   macro avg       0.57      0.50      0.50    138762
weighted avg       0.96      0.98      0.97    138762



## RF with class_weight='balanced' — sepsis is ~4% positive, more imbalanced than NTSB's fatal-accident rate

In [27]:
rf_balanced = run_rf_experiment(
    "rf_balanced",
    RandomForestClassifier(random_state=42, n_jobs=-1, class_weight="balanced"),
    X_train, y_train, X_val, y_val,
)

              precision    recall  f1-score   support

           0       0.98      1.00      0.99    135668
           1       0.14      0.00      0.00      3094

    accuracy                           0.98    138762
   macro avg       0.56      0.50      0.50    138762
weighted avg       0.96      0.98      0.97    138762



## Feature importance — same free diagnostic RF gave you in month 3

In [28]:
importances = pd.Series(rf_balanced.feature_importances_, index=feature_cols)
importances.sort_values(ascending=False).head(20)

ICULOS                 0.091394
hours_since_lactate    0.032138
Temp_max               0.019163
Age                    0.017299
Resp_mean              0.015725
HospAdmTime            0.015702
Lactate_min            0.014595
Lactate_mean           0.013858
Temp_mean              0.013572
HR_mean                0.012957
HR_max                 0.012642
Resp_min               0.012459
BUN_mean               0.011472
Resp_max               0.011338
Platelets_max          0.010853
Temp_min               0.010804
WBC_mean               0.010475
WBC_min                0.010470
WBC_max                0.010348
Platelets_mean         0.010292
dtype: float64

In [30]:
from sklearn.metrics import confusion_matrix

for name, model in [("baseline", rf_baseline), ("balanced", rf_balanced)]:
    preds = model.predict(X_val)
    tn, fp, fn, tp = confusion_matrix(y_val, preds).ravel()
    print(f"{name}: TP={tp}, FP={fp}, FN={fn}, TN={tn}")

baseline: TP=13, FP=70, FN=3081, TN=135598
balanced: TP=3, FP=19, FN=3091, TN=135649


In [ ]:
print("train positive rate:", y_train.mean())
print("val positive rate:", y_val.mean())

In [32]:
feature_cols_no_iculos = [c for c in feature_cols if c != "ICULOS"]
X_train_no_iculos = X_train[feature_cols_no_iculos]
X_val_no_iculos = X_val[feature_cols_no_iculos]

rf_no_iculos = run_rf_experiment(
    "rf_balanced_no_iculos",
    RandomForestClassifier(random_state=42, n_jobs=-1, class_weight="balanced"),
    X_train_no_iculos, y_train, X_val_no_iculos, y_val,
)

              precision    recall  f1-score   support

           0       0.98      1.00      0.99    135668
           1       0.00      0.00      0.00      3094

    accuracy                           0.98    138762
   macro avg       0.49      0.50      0.49    138762
weighted avg       0.96      0.98      0.97    138762



In [37]:
import time

random_states = [42, 0, 1, 2, 3]
results = []

for seed in random_states:
    print(f"seed={seed}")
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    train_idx, val_idx = next(splitter.split(train_dataset, groups=train_dataset["patient_id"]))
    train_split = train_dataset.iloc[train_idx]
    val_split = train_dataset.iloc[val_idx]

    X_tr, y_tr = train_split[feature_cols], train_split["label"]
    X_va, y_va = val_split[feature_cols], val_split["label"]

    for variant, weight in [("baseline", None), ("balanced", "balanced")]:
        start = time.time()
        model = RandomForestClassifier(random_state=42, n_jobs=-1, class_weight=weight)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_va)
        tn, fp, fn, tp = confusion_matrix(y_va, preds).ravel()
        elapsed = time.time() - start

        print(f"seed={seed}, variant={variant}: TP={tp} FP={fp} FN={fn} TN={tn} ({elapsed:.1f}s)")
        results.append({"split_seed": seed, "variant": variant, "TP": tp, "FP": fp, "FN": fn, "TN": tn,
                         "recall": tp/(tp+fn), "precision": tp/(tp+fp) if (tp+fp) > 0 else 0.0})

results_df = pd.DataFrame(results)

seed=42
seed=42, variant=baseline: TP=13 FP=70 FN=3081 TN=135598 (43.0s)
seed=42, variant=balanced: TP=3 FP=19 FN=3091 TN=135649 (17.8s)
seed=0
seed=0, variant=baseline: TP=1 FP=126 FN=3026 TN=135173 (40.9s)
seed=0, variant=balanced: TP=0 FP=4 FN=3027 TN=135295 (18.5s)
seed=1
seed=1, variant=baseline: TP=7 FP=65 FN=3216 TN=133241 (39.8s)
seed=1, variant=balanced: TP=4 FP=3 FN=3219 TN=133303 (18.4s)
seed=2
seed=2, variant=baseline: TP=1 FP=2 FN=2832 TN=133725 (40.2s)
seed=2, variant=balanced: TP=1 FP=6 FN=2832 TN=133721 (18.3s)
seed=3
seed=3, variant=baseline: TP=7 FP=32 FN=3328 TN=135149 (41.1s)
seed=3, variant=balanced: TP=2 FP=17 FN=3333 TN=135164 (18.3s)


In [38]:
print(results_df)

   split_seed   variant  TP   FP    FN      TN    recall  precision
0          42  baseline  13   70  3081  135598  0.004202   0.156627
1          42  balanced   3   19  3091  135649  0.000970   0.136364
2           0  baseline   1  126  3026  135173  0.000330   0.007874
3           0  balanced   0    4  3027  135295  0.000000   0.000000
4           1  baseline   7   65  3216  133241  0.002172   0.097222
5           1  balanced   4    3  3219  133303  0.001241   0.571429
6           2  baseline   1    2  2832  133725  0.000353   0.333333
7           2  balanced   1    6  2832  133721  0.000353   0.142857
8           3  baseline   7   32  3328  135149  0.002099   0.179487
9           3  balanced   2   17  3333  135164  0.000600   0.105263


In [39]:
results_df.groupby("variant")[["TP", "recall", "precision"]].agg(["mean", "std", "min", "max"])


TP                      recall                               \
         mean       std min max      mean       std      min       max   
variant                                                                  
balanced  2.0  1.581139   0   4  0.000633  0.000491  0.00000  0.001241   
baseline  5.8  5.019960   1  13  0.001831  0.001600  0.00033  0.004202   

         precision                                
              mean       std       min       max  
variant                                           
balanced  0.191183  0.220148  0.000000  0.571429  
baseline  0.154909  0.119780  0.007874  0.333333

In [40]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report

sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)

gbm = HistGradientBoostingClassifier(random_state=42)
gbm.fit(X_train, y_train, sample_weight=sample_weights)

preds = gbm.predict(X_val)
print(classification_report(y_val, preds))

              precision    recall  f1-score   support

           0       0.99      0.85      0.92    135668
           1       0.09      0.63      0.15      3094

    accuracy                           0.85    138762
   macro avg       0.54      0.74      0.54    138762
weighted avg       0.97      0.85      0.90    138762



In [41]:
from sklearn.metrics import precision_recall_curve, average_precision_score
import numpy as np

for name, model in [("rf_baseline", rf_baseline), ("rf_balanced", rf_balanced), ("gbm", gbm)]:
    probs = model.predict_proba(X_val)[:, 1]
    ap = average_precision_score(y_val, probs)
    print(f"{name}: PR-AUC = {ap:.4f}  (random baseline ≈ {y_val.mean():.4f})")

    precisions, recalls, thresholds = precision_recall_curve(y_val, probs)
    for target in [0.1, 0.2, 0.3, 0.5]:
        idx = np.argmin(np.abs(recalls - target))
        print(f"  near recall={target}: actual recall={recalls[idx]:.3f}, precision={precisions[idx]:.3f}")
    print()

rf_baseline: PR-AUC = 0.0842  (random baseline ≈ 0.0223)
  near recall=0.1: actual recall=0.098, precision=0.154
  near recall=0.2: actual recall=0.197, precision=0.139
  near recall=0.3: actual recall=0.295, precision=0.112
  near recall=0.5: actual recall=0.514, precision=0.072

rf_balanced: PR-AUC = 0.0751  (random baseline ≈ 0.0223)
  near recall=0.1: actual recall=0.104, precision=0.127
  near recall=0.2: actual recall=0.201, precision=0.122
  near recall=0.3: actual recall=0.307, precision=0.100
  near recall=0.5: actual recall=0.494, precision=0.077

gbm: PR-AUC = 0.1005  (random baseline ≈ 0.0223)
  near recall=0.1: actual recall=0.100, precision=0.142
  near recall=0.2: actual recall=0.200, precision=0.127
  near recall=0.3: actual recall=0.300, precision=0.125
  near recall=0.5: actual recall=0.500, precision=0.103



In [42]:
from sklearn.metrics import precision_recall_curve
import numpy as np

probs_val = gbm.predict_proba(X_val)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, probs_val)

# precisions/recalls have one more element than thresholds (last point has no threshold) — align them
target_recall = 0.3
idx = np.argmin(np.abs(recalls[:-1] - target_recall))
chosen_threshold = thresholds[idx]
print(f"chosen threshold: {chosen_threshold:.4f}  (val recall={recalls[idx]:.3f}, val precision={precisions[idx]:.3f})")

chosen threshold: 0.7648  (val recall=0.300, val precision=0.125)


## Final, one-time check on the held-out test set (set B)

Uses the GBM model and the threshold chosen from the validation PR curve above (0.7648, targeting recall ≈ 0.3). This is the true cross-hospital generalization check — run once, not repeatedly, and not used to pick the threshold itself (that was chosen from validation only).

In [44]:
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score

X_test = test_dataset[feature_cols]
y_test = test_dataset["label"]

test_probs = gbm.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= chosen_threshold).astype(int)

print(classification_report(y_test, test_preds))
tn, fp, fn, tp = confusion_matrix(y_test, test_preds).ravel()
print(f"TP={tp}, FP={fp}, FN={fn}, TN={tn}")
print(f"test PR-AUC: {average_precision_score(y_test, test_probs):.4f}  (random baseline ≈ {y_test.mean():.4f})")

              precision    recall  f1-score   support

           0       0.99      0.97      0.98    652555
           1       0.10      0.19      0.13      9440

    accuracy                           0.96    661995
   macro avg       0.54      0.58      0.56    661995
weighted avg       0.98      0.96      0.97    661995

TP=1807, FP=16394, FN=7633, TN=636161
test PR-AUC: 0.0659  (random baseline ≈ 0.0143)
